In [0]:
import requests
from pyspark.sql.functions import col, explode, lit

In [0]:
dbutils.widgets.text('index_no', '', 'API Pokedex Index Number')
dbutils.widgets.text('table_name', '', 'Table Name')

index_no = dbutils.widgets.get('index_no')
table_name = dbutils.widgets.get('table_name')

In [0]:
for i in range(2, 3):
  try:
    url = f'https://pokeapi.co/api/v2/region/{i}/'
    response = requests.get(url) 

    data = response.json()
  except:
    pass

In [0]:
data

In [0]:
region = data['region']['name']
pokemon_entries = data['pokemon_entries']


In [0]:
entries = (
  spark.createDataFrame(data['pokemon_entries'])
  .withColumn('pokedex_index_no', lit(int(index_no)))
  .select("pokedex_index_no", "entry_number", col("pokemon_species.name").alias("pokemon_name"), col("pokemon_species.url").alias("pokemon_url"))
)

entries.display()

# entries.write.format('delta').mode("overwrite").option('overwriteSchema', 'true').saveAsTable(f"pokemon.{table_name}")

In [0]:
spark.sql(f"""
    SELECT 
        *
    FROM 
        pokemon.{table_name}
    ORDER BY 
        entry_number
""").display()